# Classical Phase Vocoder

This is an implementation of the classical phase vocoder, based on the paper "Phase Vocoder Done Right" by Zdeněk Průša and Nicki Holighaus.

Phase vocoder time stretching follows a six-step process:
1. Import the audio file.
2. Perform the STFT.
3. Decompose the complex STFT matrix into magnitude and phase matrices.
4. Determine time stretch factor $\alpha$.
5. Compute an updated phase matrix corresponding to $\alpha$.
6. Perform an inverse STFT, using a new hop size corresponding to $\alpha$.

The starter code below contains functionality for performing the STFT.

In [ ]:
import soundfile as sf
import numpy as np

def stft_frames(audio_frames, fft_size, hop_size):
    """
    Gets the number of STFT frames for given STFT parameters
    :param audio_frames: The number of audio frames
    :param fft_size: The FFT size
    :param hop_size: The hop size
    :return: The number of STFT frames
    """
    return 1 + (audio_frames - fft_size) // hop_size

def audio_frames(stft_frames, fft_size, hop_size):
    """
    Gets the number of audio frames for given STFT parameters
    :param stft_frames: The number of STFT frames
    :param fft_size: The FFT size
    :param hop_size: The hop size
    :return: The number of audio frames
    """
    return (stft_frames - 1) * hop_size + fft_size

def stft( input_sound, fft_size, hop_size, window):
    """
    Compute the short-time Fourier transform of a sound.
    Args:
        input_sound: 1D numpy array containing the sound samples
        fft_size: Size of the FFT to compute (number of frequency bins)
        hop_size: Number of samples to advance between successive frames
        window: 1D numpy array containing the analysis window (length = dft_size)
    Returns:
        f: 2D numpy array (frequencies x time) containing the complex-valued STFT
    """
    num_frames = stft_frames(np.max(input_sound.shape), fft_size, hop_size)
    spectrogram = np.zeros((fft_size // 2 + 1, num_frames), dtype=np.complex64)
    for frame in range(num_frames):
        start = frame * hop_size
        end = start + fft_size
        segment = input_sound[start:end] * window
        spectrum = np.fft.rfft(segment, n=fft_size)
        spectrogram[:, frame] = spectrum

    # Return a complex-valued spectrogram (frequencies x time)    
    return spectrogram

def istft( stft_output, fft_size, hop_size, window, norm=None):
    """Compute the inverse short-time Fourier transform of a spectrogram.
    Args:
        stft_output: 2D numpy array (frequencies x time) containing the complex-valued STFT
        fft_size: Size of the FFT that was computed (number of frequency bins)
        hop_size: Number of samples advanced between successive frames
        window: 1D numpy array containing the synthesis window (length = dft_size)
    Returns:
        x: 1D numpy array containing the reconstructed sound samples
    """
    number_frames = stft_output.shape[1]
    x = np.zeros((audio_frames(number_frames, fft_size, hop_size),))
    
    for frame in range(number_frames):
        start = frame * hop_size
        end = start + fft_size
        windowed_frame = np.fft.irfft(stft_output[:, frame], n=fft_size)
        x[start:end] += windowed_frame * window
    
    if norm is not None:
        return x / (norm + 1e-8)
    else:
        return x

def compute_window_norm(analysis_window, synthesis_window, hop_size, output_length):
    """
    Computes a window norm for the ISTFT
    :param analysis_window: The analysis window
    :param synthesis_window: The synthesis window
    :param num_frames: The number of frames in the STFT
    :return: The window norm array
    """
    analysis_norm = np.zeros((output_length,))
    synthesis_norm = np.zeros((output_length,))
    i = 0
    while (j := i + analysis_window.shape[-1]) <= output_length:
        analysis_norm[i:j] += analysis_window
        i += hop_size
    i = 0
    while (j := i + synthesis_window.shape[-1]) <= output_length:
        synthesis_norm[i:j] += synthesis_window
        i += hop_size
    norm = analysis_norm * synthesis_norm
    return norm

def sound_player( x, rate=8000, label=''):
    """
    Sound player courtesy of Prof. Minje Kim in UIUC CS department
    :param x: The NumPy array to play
    :param rate: The sample rate
    :param label: An optional text label for the sound file
    """
    from IPython.display import display, Audio, HTML
    display( HTML( 
    '<style> table, th, td {border: 0px; }</style> <table><tr><td>' + label + 
    '</td><td>' + Audio( x, rate=rate)._repr_html_()[3:] + '</td></tr></table>'
    ))

**(step 1)** Now that we have our starter code, we will select an audio file:

In [ ]:
!wget -O "synth.wav" https://getsamplefiles.com/download/wav/sample-4.wav
!wget -O "dream.wav" https://github.com/voxserv/audio_quality_testing_samples/raw/refs/heads/master/mono_44100/156550__acclivity__a-dream-within-a-dream.wav

In [ ]:
audio, sr = sf.read("synth.wav")
# or, to use a speech example:
# audio, sr = sf.read("dream.wav")
if audio.ndim > 1:
  audio = audio.sum(axis=1)
print(audio.shape)
sound_player(audio, sr, "original sound")

(220500,)


original sound,Your browser does not support the audio element.


**(step 2)** and perform the STFT:

In [7]:
FFT_SIZE = 2048
HOP_SIZE = FFT_SIZE//4  # for a FFT size of 2048, the hop size is 512
window = np.hanning(FFT_SIZE)
analysis_data = stft(audio, FFT_SIZE, HOP_SIZE, window)
print(f"Shape of STFT matrix: {analysis_data.shape}")

Shape of STFT matrix: (1025, 427)


**(step 3)** and decompose the complex STFT matrix into magnitude and phase matrices:

In [8]:
# The magnitude spectrogram
analysis_mag_mx = np.abs(analysis_data)
# The phase spectrogram
analysis_phase_mx = np.angle(analysis_data)

**(step 4)** and determine our time stretch factor $\alpha$:

In [9]:
alpha = 1.5

**(step 5)** Now we need to compute our updated phase matrix. We will use Průša and Holighaus's formulas 7 and 13 for this:
$$
\phi_{\textrm{s}} (m,n) = \phi_{\textrm{s}} (m, n-1) + a_{\textrm{s}} (\Delta_{\textrm{t}} \phi_{\textrm{a}})(m, n) \tag{7}
$$
$$
\Delta_{\textrm{t,back}}\phi_{\textrm{a}} (m, n) = \frac{1}{a_{\textrm{a}}}\left[ \phi_{\textrm{a}}(m,n) - \phi_{\textrm{a}}(m,n-1) - \frac{2 \pi ma_{\textrm{a}}}{M}\right]_{2\pi} + \frac{2 \pi m}{M} \tag{13}
$$
Note that:
- $m$ is the frequency bin index and $n$ is the STFT frame index,
- $a_{\textrm{a}}$ is the analysis hop size and $a_{\textrm{s}}$ is the synthesis hop size,
- $M$ is the FFT size.
- $[\cdot]_{2\pi}$ means the *principal argument* of the angle: we are mapping the angle to the range $[0, 2\pi)$ using formula 12.

In [28]:
def princarg(angle):
    """
    Computes the principal argument of an angle
    :param angle: The angle
    :return: The principal argument
    """
    return angle - 2 * np.pi * np.round(angle / (2 * np.pi))

def compute_synthesis_phase_frame(current_aframe, prev_aframe, prev_sframe, analysis_hop, synthesis_hop, m):
    """
    Computes the phases of a synthesis frame
    :param current_aframe: The current analysis phase frame
    :param prev_aframe: The previous analysis phase frame
    :param prev_sframe: The previous synthesis phase frame
    :param analysis_hop: The analysis hop size (# samples)
    :param synthesis_hop: The synthesis hop size (# samples)
    :param m: The FFT size
    :return: The new synthesis phase frame
    """
    bin_ratios = 2 * np.pi * np.linspace(0, m//2, m//2+1) / m
    deriv_t = 1/analysis_hop * princarg(current_aframe - prev_aframe - analysis_hop * bin_ratios) + bin_ratios
    return prev_sframe + synthesis_hop * deriv_t

For the first synthesis phase frame, we will simply reuse the first analysis phase frame. All other synthesis phase frames will propagate from this.

In [29]:
SYNTHESIS_HOP_SIZE = int(HOP_SIZE * alpha)
synthesis_phase_mx = [analysis_phase_mx[:, 0]]
for i in range(1, analysis_phase_mx.shape[-1]):
    synthesis_phase_mx.append(compute_synthesis_phase_frame(
        analysis_phase_mx[:, i],
        analysis_phase_mx[:, i-1],
        synthesis_phase_mx[-1],
        HOP_SIZE,
        SYNTHESIS_HOP_SIZE,
        FFT_SIZE
    ))
synthesis_phase_mx = np.vstack(synthesis_phase_mx).T

**(step 6)** Now we perform an inverse STFT with the synthesis hop size:

In [ ]:
synthesis_mx = analysis_mag_mx * np.exp(1j * synthesis_phase_mx)
synthesis_norm = compute_window_norm(window, window, SYNTHESIS_HOP_SIZE, audio_frames(synthesis_mx.shape[-1], FFT_SIZE, SYNTHESIS_HOP_SIZE))
out_audio = istft(synthesis_mx, FFT_SIZE, SYNTHESIS_HOP_SIZE, window, synthesis_norm)
sound_player(out_audio, sr, "Time-stretched audio")

Time-stretched audio,Your browser does not support the audio element.


If you don't like the clicks you hear at the beginning or end of the sound, those are easy to fix.

You can also make the output sound louder by multiplying `out_audio` by some number, like 2.0. BE CAREFUL if you choose to do this--you can get very loud sounds in your headphones.

In [ ]:
fader = np.hanning(FFT_SIZE)
out_audio[:FFT_SIZE//2] *= fader[:FFT_SIZE//2]
out_audio[-(FFT_SIZE//2):] *= fader[-(FFT_SIZE//2):]
sound_player(out_audio, sr, "Time-stretched audio")

Time-stretched audio,Your browser does not support the audio element.


Try doing this time stretch with different $\alpha$ values and see what happens. You may notice stuttering in the sound due to transient issues. Průša and Holighaus's RTPGHI algorithm helps to address transient issues.